In [1]:
from pyspark.sql.functions import col, to_date, trim
from pyspark.sql.types import IntegerType

# Read from Bronze (additional lakehouse - needs prefix)
df_inv = spark.read.format("delta").table("lh_bronze.dbo.server_inventory")

# Cast and clean
df_inv_silver = df_inv \
    .withColumn("cpu_cores", col("cpu_cores").cast(IntegerType())) \
    .withColumn("ram_gb", col("ram_gb").cast(IntegerType())) \
    .withColumn("install_date", to_date(col("install_date"), "yyyy-MM-dd")) \
    .withColumn("status", trim(col("status"))) \
    .dropDuplicates(["server_id"])

# Write to Silver (primary attached lakehouse - no prefix needed)
df_inv_silver.write.format("delta").mode("overwrite").saveAsTable("server_inventory")

print(f"✅ server_inventory: {df_inv_silver.count()} rows written to lh_silver")

StatementMeta(, df19ba97-8b21-4f9f-8266-e88be373baee, 3, Finished, Available, Finished, False)

✅ server_inventory: 25 rows written to lh_silver


In [2]:
from pyspark.sql.functions import col, to_date, trim
from pyspark.sql.types import FloatType, BooleanType

# Read from Bronze
df_inc = spark.read.format("delta").table("lh_bronze.dbo.incident_log")

# Cast and clean
df_inc_silver = df_inc \
    .withColumn("incident_date", to_date(col("incident_date"), "yyyy-MM-dd")) \
    .withColumn("resolution_time_hrs", col("resolution_time_hrs").cast(FloatType())) \
    .withColumn("resolved", col("resolved").cast(BooleanType())) \
    .withColumn("severity", trim(col("severity"))) \
    .withColumn("incident_type", trim(col("incident_type"))) \
    .dropDuplicates(["incident_id"])

# Referential integrity check - join against Silver (already deduped)
df_valid_servers = spark.read.format("delta").table("server_inventory").select("server_id")
df_inc_silver = df_inc_silver.join(df_valid_servers, on="server_id", how="inner")

# Write to Silver
df_inc_silver.write.format("delta").mode("overwrite").saveAsTable("incident_log")

print(f"✅ incident_log: {df_inc_silver.count()} rows written to lh_silver")

StatementMeta(, df19ba97-8b21-4f9f-8266-e88be373baee, 4, Finished, Available, Finished, False)

✅ incident_log: 55 rows written to lh_silver


In [3]:
from pyspark.sql.functions import col, to_timestamp, trim
from pyspark.sql.types import FloatType

# Set legacy time parser to handle single-digit hours
spark.conf.set("spark.sql.legacy.timeParserPolicy", "LEGACY")

# Read from Bronze
df_cap = spark.read.format("delta").table("lh_bronze.dbo.capacity_utilization")

# Cast and clean
df_cap_silver = df_cap \
    .withColumn("timestamp", to_timestamp(col("timestamp"), "yyyy-MM-dd H:mm:ss")) \
    .withColumn("cpu_pct", col("cpu_pct").cast(FloatType())) \
    .withColumn("ram_pct", col("ram_pct").cast(FloatType())) \
    .withColumn("storage_pct", col("storage_pct").cast(FloatType())) \
    .withColumn("network_mbps", col("network_mbps").cast(FloatType())) \
    .withColumn("power_watts", col("power_watts").cast(FloatType())) \
    .dropDuplicates(["record_id"]) \
    .filter(col("cpu_pct").between(0, 100)) \
    .filter(col("ram_pct").between(0, 100)) \
    .filter(col("storage_pct").between(0, 100))

# Referential integrity check
df_valid_servers = spark.read.format("delta").table("server_inventory").select("server_id")
df_cap_silver = df_cap_silver.join(df_valid_servers, on="server_id", how="inner")

# Write to Silver
df_cap_silver.write.format("delta").mode("overwrite").saveAsTable("capacity_utilization")

print(f"✅ capacity_utilization: {df_cap_silver.count()} rows written to lh_silver")

StatementMeta(, df19ba97-8b21-4f9f-8266-e88be373baee, 5, Finished, Available, Finished, False)

✅ capacity_utilization: 136 rows written to lh_silver


In [4]:
from pyspark.sql.functions import col, to_date
from pyspark.sql.types import FloatType, IntegerType

# Same pattern as other Bronze tables
df_weather = spark.read.format("delta").table("lh_bronze.dbo.weather_ashburn")

# Cast and clean
df_weather_silver = df_weather \
    .withColumn("date", to_date(col("date"), "yyyy-MM-dd")) \
    .withColumn("temp_max_f", col("temp_max_f").cast(FloatType())) \
    .withColumn("temp_min_f", col("temp_min_f").cast(FloatType())) \
    .withColumn("precipitation_in", col("precipitation_in").cast(FloatType())) \
    .withColumn("windspeed_max_mph", col("windspeed_max_mph").cast(FloatType())) \
    .withColumn("weather_code", col("weather_code").cast(IntegerType())) \
    .dropDuplicates(["date", "location"])

# Write to Silver
df_weather_silver.write.format("delta").mode("overwrite").saveAsTable("weather_ashburn")

print(f"✅ weather_ashburn: {df_weather_silver.count()} rows written to lh_silver")

StatementMeta(, df19ba97-8b21-4f9f-8266-e88be373baee, 6, Finished, Available, Finished, False)

✅ weather_ashburn: 14 rows written to lh_silver
